<a href="https://colab.research.google.com/github/gokadagaya/Assignments-YSE-0/blob/master/Pandas_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Load and Inspect the Data

In [5]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)
print(df.info)
missing_values=df.isna().sum()
print(missing_values)

<bound method DataFrame.info of     transaction_id       date region product_category  sales_amount  quantity  \
0                1 2024-10-01  North      Electronics        1200.0       2.0   
1                2 2024-10-02  South         Clothing         450.0       5.0   
2                3 2024-10-03   East             None         890.0       3.0   
3                4 2024-10-04   West            Books           NaN       1.0   
4                5 2024-10-05   None      Electronics        1500.0       NaN   
5                6 2024-10-06  North             Home         670.0       4.0   
6                7 2024-10-07  South         Clothing           NaN       2.0   
7                8 2024-10-08   None            Books         340.0       3.0   
8                9 2024-10-09   East      Electronics        2100.0       1.0   
9               10 2024-10-10   West             None         780.0       5.0   
10              11 2024-10-11  North             Home         560.0       3.0

Task 2: Handle Missing Values


In [9]:
df[['region', 'product_category']] = df[['region', 'product_category']].fillna(
    df[['region', 'product_category']].mode().iloc[0]
)
df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())
df['quantity'] = df['quantity'].fillna(method='ffill')
df['customer_age'] = df['customer_age'].fillna(df['customer_age'].mean()).round().astype(int)
df = df.dropna(subset=['payment_method'])

vfy=df.isna().sum()
vfy

/tmp/ipython-input-1214/3425397315.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['quantity'] = df['quantity'].fillna(method='ffill')


,0
transaction_id,0
date,0
region,0
product_category,0
sales_amount,0
quantity,0
customer_age,0
payment_method,0


Task 3: GroupBy Analysis


In [14]:
total_sales=df.groupby('region')['sales_amount'].sum()
print(total_sales)

region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64


In [15]:
AVG_SALES=df.groupby('product_category')['sales_amount'].mean()
print(AVG_SALES)


product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64


In [16]:
combine= df.groupby(['region', 'product_category'])[['sales_amount', 'quantity']].sum().reset_index()
print(combine)


  region product_category  sales_amount  quantity
0   East            Books         800.0       4.0
1   East         Clothing         890.0       3.0
2   East      Electronics        2100.0       1.0
3  North         Clothing         510.0       3.0
4  North      Electronics        2700.0       3.0
5  North             Home        3250.0      12.0
6  South         Clothing        1900.0       9.0
7   West            Books         725.0       1.0
8   West         Clothing         780.0       5.0
9   West      Electronics         725.0       1.0


In [17]:
top_3 = combine.sort_values('sales_amount', ascending=False).head(3)
print(top_3)

  region product_category  sales_amount  quantity
5  North             Home        3250.0      12.0
4  North      Electronics        2700.0       3.0
2   East      Electronics        2100.0       1.0


Task 4: Custom Aggregation


In [22]:
def sales_range(x):
    return x.max() - x.min()
sales_region = df.groupby('region')['sales_amount'].agg(sales_range)
print(sales_region)
region_summary = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean', 'max'],
    'quantity': ['sum', 'min']
})
print(region_summary)

region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64
       sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0
